# Citation networks and simulation variants

Compare an empirical citation network with the **equalize** and **randomization**
methods from `model.equality_study.build_setting`, the same builder used by the
simulation notebooks. Both methods keep the node and edge counts fixed.
The simulations enable equalization by default; randomization is shown here as
an additional comparison. Equalization uses uniform degree targets while
restoring clustering toward the original network's average.

Choose PUD, tobacco, or ego depletion below. Variant parameters and seeds follow
the simulation workflow; this notebook builds graphs without running simulations.
All panels share the original network's coordinates so that edge changes are
visible. Figures are saved under `results/figures/`.

The core plots use NetworkX and Matplotlib from the project environment.
Graphistry with igraph and the legacy edge-bundling helper are optional; when
unavailable, the notebook uses a community-weighted spring layout and straight
edges. The final power-law analysis is also optional.

In [ ]:
from pathlib import Path
import ast
import sys
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import networkx as nx
from IPython.display import display
from networkx.algorithms.community import louvain_communities

# Resolve project paths whether Jupyter starts here or in a subdirectory.
PROJECT_ROOT = next(
    directory for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / 'networks' / 'citation_data').is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model.equality_study import build_setting, variant_sequences, variant_seeds
from utils.network_utils import calculate_degree_gini, network_statistics

try:
    import colormaps
    COLORMAP = colormaps.redp1
except ImportError:
    COLORMAP = plt.cm.viridis

try:
    import graphistry
    import igraph
except ImportError:
    graphistry = None

try:
    from utils.network_plot_utils import draw_citation_graph
except ModuleNotFoundError as exc:
    if exc.name not in {'NetworkInequality', 'NetworkInequality.edgebundling'}:
        raise
    draw_citation_graph = None

plt.style.use('default')
sns.set_theme(style='white')
SEED = 42  # Layout seed; variant seeds are configured separately below.
BUNDLE_EDGES = draw_citation_graph is not None

## Load the base network


In [ ]:
NETWORK_KEY = 'pud'  # 'pud', 'tobacco', or 'ego'
NETWORK_TITLES = {
    'pud': 'Peptic ulcer disease',
    'tobacco': 'Tobacco and health',
    'ego': 'Ego depletion',
}
NETWORK_TITLE = NETWORK_TITLES[NETWORK_KEY]
DATA_PATH = PROJECT_ROOT / 'networks' / 'citation_data' / f'{NETWORK_KEY}_network.pkl'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with DATA_PATH.open('rb') as f:
    base_network = pickle.load(f)

print(f'{NETWORK_TITLE}: {base_network.number_of_nodes():,} nodes, '
      f'{base_network.number_of_edges():,} edges')

## Optional: attach average publication year (PUD only, pre-1978)

If `results/pud_avg_pub_year_pre1978.csv` exists, it is loaded automatically for
PUD. To recompute it with OpenAlex, set `RUN_OPENALEX = True`. The default plots
work offline and use a flat node color when metadata is unavailable.

In [ ]:
META_PATH = PROJECT_ROOT / "results" / "pud_avg_pub_year_pre1978.csv"
RUN_OPENALEX = False

rebuild_metadata = False
df_avg = None
if RUN_OPENALEX and NETWORK_KEY != "pud":
    raise ValueError("The pre-1978 metadata query is only configured for PUD.")
if NETWORK_KEY == "pud" and META_PATH.exists() and not rebuild_metadata:
    df_avg = pd.read_csv(META_PATH)
elif RUN_OPENALEX:
    from pyalex import Works
    import pyalex
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from collections import defaultdict
    import math
    from tqdm import tqdm

    pyalex.config.email = "you@example.com"

    raw_nodes = list(base_network.nodes())
    author_ids = []
    for x in raw_nodes:
        s = str(x).strip()
        if s.startswith("https://openalex.org/"):
            s = s.rsplit("/", 1)[-1]
        if s and s[0].upper() == "A":
            author_ids.append(s.upper())

    author_set = set(author_ids)
    print(f"{len(author_ids)} authors")

    def chunks(lst, n=100):
        for i in range(0, len(lst), n):
            yield lst[i : i + n]

    def fetch_chunk(aids):
        id_str = "|".join(aids)
        pager = (
            Works()
            .filter(author={"id": id_str}, to_publication_date="1977-12-31")
            #.select(["id", "publication_year", "authorships"])
            .paginate(per_page=200)
        )
        out = []
        for page in pager:
            out.extend(page)
            
            
        #print(out)
        return out

    sum_years = defaultdict(int)
    count_works = defaultdict(int)
    coauthors = defaultdict(set)
    

    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = {ex.submit(fetch_chunk, ch): ch for ch in chunks(author_ids, 100)}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Chunks"):
            for w in fut.result():
                y = w.get("publication_year")
                if not isinstance(y, int):
                    continue
                # Collect all author ids in this work that are in our set
                work_authors_in_set = []
                # Collect all author ids in this work (regardless of whether in our set)
                all_work_authors = []
                for au in (w.get("authorships") or []):
                    aid = (au.get("author") or {}).get("id")
                    if not aid:
                        continue
                    aid_short = aid.rsplit("/", 1)[-1]
                    all_work_authors.append(aid_short)
                    if aid_short in author_set:
                        work_authors_in_set.append(aid_short)
                
                # Update stats and coauthors for each author in the work
                for aid_short in work_authors_in_set:
                    sum_years[aid_short] += y
                    count_works[aid_short] += 1
                    # Add all authors from this work as coauthors (not just those in our set)
                    for other_aid in all_work_authors:
                        if other_aid != aid_short:
                            coauthors[aid_short].add(other_aid)

    rows = []
    for aid in author_ids:
        n = count_works.get(aid, 0)
        avg = (sum_years[aid] / n) if n else math.nan
        rows.append({
            "author_id": aid,
            "works_pre1978": n,
            "avg_pub_year_pre1978": avg,
            "coauthors": list(coauthors.get(aid, []))
        })

    df_avg = pd.DataFrame(rows).sort_values(
        ["works_pre1978", "avg_pub_year_pre1978"], ascending=[False, True]
    )
    df_avg.to_csv(META_PATH, index=False)
else:
    print("No avg_pub_year_pre1978 metadata found; nodes will render in a flat color.")

if df_avg is not None:
    for _, row in df_avg.iterrows():
        full_author_id = f"https://openalex.org/{row['author_id']}"
        if full_author_id in base_network:
            base_network.nodes[full_author_id]["avg_pub_year_pre1978"] = row[
                "avg_pub_year_pre1978"
            ]
            base_network.nodes[full_author_id]["works_pre1978"] = row["works_pre1978"]
            base_network.nodes[full_author_id]["coauthors"] = row["coauthors"] if isinstance(row["coauthors"], list) else ast.literal_eval(row["coauthors"]) if isinstance(row["coauthors"], str) else []

In [ ]:
base_network.nodes(data=True)

In [ ]:
# Check if A5113573464 is in the coauthor nodes
target_author = "https://openalex.org/A5113573464"
if target_author in base_network.nodes:
    print(f"{target_author} is in the network")
    print(f"Node data: {base_network.nodes[target_author]}")
else:
    print(f"{target_author} is NOT in the network")

# Also check without the full URL prefix
target_short = "A5113573464"
matching_nodes = [n for n in base_network.nodes if target_short in base_network.nodes[n].get('coauthors', [])]
print(f"Nodes containing '{target_short}': {matching_nodes}")

## Helpers


In [ ]:
HAMMER_PARAMS = {
    "accuracy": 500,
    "advect_iterations": 50,
    "batch_size": 20000,
    "decay": 0.009,
    "initial_bandwidth": 1.2,
    "iterations": 6,
    "max_segment_length": 0.016,
    "min_segment_length": 0.008,
    "tension": 1.4,
}


def compute_cluster_layout(G, seed=SEED, niter=500):
    """Community-weighted layout without changing the graph being visualized."""
    comms = louvain_communities(G, seed=seed)
    cluster = {n: i for i, community in enumerate(comms) for n in community}
    layout_graph = G.copy()
    nx.set_node_attributes(layout_graph, cluster, "cluster")

    if graphistry is not None:
        igraph.set_random_number_generator(random.Random(seed))
        g = graphistry.bind(source="src", destination="dst").from_networkx(layout_graph)
        g_layout = g.modularity_weighted_layout(
            community_col="cluster",
            cross_community_weight=0.1,
            same_community_weight=5,
        )
        g_layout = g_layout.layout_igraph(
            "fr", directed=False, params={"weights": "weight", "niter": niter}
        )
        return pd.DataFrame(g_layout._nodes).set_index("n")

    # Apply the same community weights with the core NetworkX dependency.
    layout_graph = layout_graph.to_undirected()
    for u, v in layout_graph.edges():
        layout_graph.edges[u, v]["weight"] = 5 if cluster[u] == cluster[v] else 0.1
    positions = nx.spring_layout(layout_graph, seed=seed, iterations=niter, weight="weight")
    return pd.DataFrame.from_dict(positions, orient="index", columns=["x", "y"])


def plot_network_panel(
    ax,
    G,
    position_df,
    title,
    *,
    cmap=COLORMAP,
    norm=None,
    min_year=1925,
    show_colorbar=False,
    title_y=1.02,
):
    node_ids = list(G.nodes())
    xy = position_df.loc[node_ids, ["x", "y"]].to_numpy()
    degrees = np.array([G.degree(n) for n in node_ids])
    node_sizes = degrees * 3 + 6

    if BUNDLE_EDGES and draw_citation_graph is not None:
        render_graph = G.copy()
        for node, (x, y) in zip(node_ids, xy):
            render_graph.nodes[node].update(X=x, Y=y, color="#808080")
        draw_citation_graph(
            render_graph, ax=ax, bundle_edges=True, alpha=0.3,
            hammer_kwargs=HAMMER_PARAMS,
        )
    else:
        positions = dict(zip(node_ids, xy))
        # Suppress arrowheads for readability; the underlying graphs stay directed.
        nx.draw_networkx_edges(
            G, positions, ax=ax, arrows=False, edge_color="gray", width=0.6, alpha=0.45,
        )

    color_values = np.array(
        [G.nodes[n].get("avg_pub_year_pre1978", np.nan) for n in node_ids], dtype=float
    )
    if np.isfinite(color_values).any():
        color_values = np.clip(color_values, min_year, None)
        ax.scatter(
            xy[:, 0],
            xy[:, 1],
            s=node_sizes,
            c=color_values,
            cmap=cmap,
            norm=norm,
            alpha=0.5,
            linewidth=0.5,
            zorder=2,
        )
        if show_colorbar and norm is not None:
            sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
            plt.colorbar(sm, ax=ax, shrink=0.6)
    else:
        ax.scatter(
            xy[:, 0],
            xy[:, 1],
            s=node_sizes,
            c="#62778a",
            alpha=0.75,
            linewidth=0.5,
            zorder=2,
        )

    ax.set_title(title, y=title_y)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")

## Build variants with the simulation workflow

`MASTER_SEED`, `VARIANT_INDEX`, and `PROPORTION_EDGES_MAX` select the examples.
The builder draws the edited fraction uniformly between zero and the cap
(10% by default), separately for each method. Node order and integer labels
match the simulation notebooks; the original author ID is retained as an attribute.
`uncertainty` and `n_experiments` are simulation metadata and do not affect these
graph edits. No model runs are launched.

These are regenerated examples using the current editing code. Matching an old
result also requires the original network, settings, and editing-code version.

In [ ]:
G_original = nx.convert_node_labels_to_integers(base_network, label_attribute='author_id')

In [ ]:
MASTER_SEED = 20260723  # Same default as the simulation notebooks.
VARIANT_INDEX = 0
PROPORTION_EDGES_MAX = 0.1
VARIANT_CONFIGS = [('Equalized', 'equalize'), ('Randomized', 'randomization')]

G_variants = []
variant_settings = {}
original_edges = set(G_original.edges())
summary_rows = [{
    'variant': 'Original', 'method': 'original', 'variation_seed': None,
    'proportion_edges': 0.0, 'requested_edits': 0, 'edges_removed': 0, 'edges_added': 0,
    'n_agents': G_original.number_of_nodes(), 'n_edges': G_original.number_of_edges(),
    **network_statistics(G_original),
}]

for label, method in VARIANT_CONFIGS:
    sequence = variant_sequences(MASTER_SEED, NETWORK_KEY, method, VARIANT_INDEX + 1)[VARIANT_INDEX]
    variation_seed, _ = variant_seeds(sequence, n_runs=0)
    setting = build_setting(
        G_original, method, variation_seed,
        uncertainty=0.001, n_experiments=1000,
        proportion_edges_max=PROPORTION_EDGES_MAX,
    )
    G_variant = setting['network']
    assert set(G_variant) == set(G_original), 'Variant changed the node set'
    assert G_variant.number_of_edges() == G_original.number_of_edges(), 'Density changed'
    assert not list(nx.selfloop_edges(G_variant)), 'Variant introduced self-loops'
    assert set(G_original.edges()) == original_edges, 'Base graph was modified'

    G_variants.append((label, G_variant))
    variant_settings[label] = setting
    edited_edges = set(G_variant.edges())
    summary_rows.append({
        'variant': label, 'method': method,
        **{key: value for key, value in setting.items() if key != 'network'},
        'requested_edits': int(G_original.number_of_edges() * setting['proportion_edges']),
        'edges_removed': len(original_edges - edited_edges),
        'edges_added': len(edited_edges - original_edges),
    })

variant_summary = pd.DataFrame(summary_rows).set_index('variant')
variant_summary['variation_seed'] = variant_summary['variation_seed'].astype('Int64')
display(variant_summary)

## Compute layouts


In [ ]:
layout_original = compute_cluster_layout(G_original)
# Use identical coordinates and bounds for every network panel.

color_values = np.array(
    [G_original.nodes[n].get("avg_pub_year_pre1978", np.nan) for n in G_original.nodes()],
    dtype=float,
)
MIN_YEAR = 1925
if np.isfinite(color_values).any():
    color_values = np.clip(color_values, MIN_YEAR, None)
    vmax = np.nanmax(color_values)
    COLOR_NORM = plt.Normalize(vmin=MIN_YEAR, vmax=vmax)
else:
    COLOR_NORM = None

## Facet plot


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
panels = [('Original', G_original), *G_variants]
xy = layout_original[['x', 'y']].to_numpy()
padding = np.maximum(np.ptp(xy, axis=0) * 0.08, 0.05)

for index, (ax, (label, G)) in enumerate(zip(axes, panels)):
    row = variant_summary.loc[label]
    title = f"{chr(97 + index)}) {label}"
    if label != 'Original':
        title += f" ({row['proportion_edges']:.1%} requested)"
    plot_network_panel(
        ax, G, layout_original, title, norm=COLOR_NORM,
        show_colorbar=(index == 0),
    )
    ax.set_xlim(xy[:, 0].min() - padding[0], xy[:, 0].max() + padding[0])
    ax.set_ylim(xy[:, 1].min() - padding[1], xy[:, 1].max() + padding[1])
    ax.text(
        0.5, -0.04,
        f"{G.number_of_nodes()} nodes · {G.number_of_edges()} edges · "
        f"out-degree Gini {row['degree_gini_coefficient']:.3f}",
        ha='center', transform=ax.transAxes, fontsize=10,
    )

fig.suptitle(f'{NETWORK_TITLE}: simulation network edits', fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.9))
FIGURE_PATH = OUTPUT_DIR / f'{NETWORK_KEY}_simulation_network_variants.pdf'
fig.savefig(FIGURE_PATH, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {FIGURE_PATH}')

In [ ]:
fig = plt.figure(figsize=(10, 15))

gs = fig.add_gridspec(3, 2, width_ratios=[1, 1], height_ratios=[1, 1, 1], wspace=0.05, hspace=0.05)
ax_main = fig.add_subplot(gs[0:2, :])
ax_left = fig.add_subplot(gs[2, 0])
ax_right = fig.add_subplot(gs[2, 1])

plot_network_panel(
    ax_main,
    G_original,
    layout_original,
    f"Original {NETWORK_TITLE} citation network",
    norm=COLOR_NORM,
    show_colorbar=True,
)

plot_network_panel(
    ax_left,
    G_variants[0][1],
    layout_original,
    G_variants[0][0],
    norm=COLOR_NORM,
)

plot_network_panel(
    ax_right,
    G_variants[1][1],
    layout_original,
    G_variants[1][0],
    norm=COLOR_NORM,
)

# fig.su
# ptitle("PUD network variants", y=0.98)


plt.savefig(OUTPUT_DIR / f"{NETWORK_KEY}_simulation_network_variants_tall.pdf", dpi=300)

plt.show()

# Create separate plot for original network only
fig_original, ax_original = plt.subplots(figsize=(10, 10))
plot_network_panel(
    ax_original,
    G_original,
    layout_original,
    f"Original {NETWORK_TITLE} citation network",
    norm=COLOR_NORM,
    show_colorbar=True,
)
plt.savefig(OUTPUT_DIR / f"{NETWORK_KEY}_network_original.pdf", dpi=300)
plt.show()

# Create separate plot for variant networks
fig_variants, (ax_v1, ax_v2) = plt.subplots(1, 2, figsize=(12, 6))
plot_network_panel(
    ax_v1,
    G_variants[0][1],
    layout_original,
    G_variants[0][0],
    norm=COLOR_NORM,
)
plot_network_panel(
    ax_v2,
    G_variants[1][1],
    layout_original,
    G_variants[1][0],
    norm=COLOR_NORM,
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{NETWORK_KEY}_network_changed.pdf", dpi=300)
plt.show()

In [ ]:
# Replicate Figure 2: 10 person cycle, wheel, and complete graph
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))

# Create the three networks
n = 10

# Cycle graph
G_cycle = nx.cycle_graph(n)
pos_cycle = nx.circular_layout(G_cycle)

# Wheel graph
G_wheel = nx.wheel_graph(n)
# Create circular layout for outer nodes (1 through n)
outer_nodes = list(range(1, n + 1))
pos_wheel = nx.circular_layout(G_wheel.subgraph(outer_nodes))
# Set center node (0) at origin
pos_wheel[0] = np.array([0.0, 0.0])


# Complete graph
G_complete = nx.complete_graph(n)
pos_complete = nx.circular_layout(G_complete)

# Generate random values for nodes
np.random.seed(42)  # For reproducibility
values_cycle = np.full(n, 0.3)
values_wheel = np.full(n, 0.5)
values_complete = np.full(n, 0.7)

# Plot cycle
nx.draw_networkx_nodes(G_cycle, pos_cycle, ax=axes[0], node_color=values_cycle, node_size=300, cmap=COLORMAP, vmin=0, vmax=1)
nx.draw_networkx_edges(G_cycle, pos_cycle, ax=axes[0], edge_color='gray', width=2, alpha=0.5)
# Add invisible scatterplot for all nodes
x_cycle = [pos_cycle[node][0] for node in G_cycle.nodes()]
y_cycle = [pos_cycle[node][1] for node in G_cycle.nodes()]
axes[0].scatter(x_cycle, y_cycle, s=300, alpha=0)
axes[0].set_title("Cycle", fontsize=14)
axes[0].axis('off')
axes[0].set_aspect('equal')

# Plot wheel
nx.draw_networkx_nodes(G_wheel, pos_wheel, ax=axes[1], node_color=values_wheel, node_size=300, cmap=COLORMAP, vmin=0, vmax=1)
nx.draw_networkx_edges(G_wheel, pos_wheel, ax=axes[1], edge_color='gray', width=2, alpha=0.5)
# Add invisible scatterplot for all nodes
x_wheel = [pos_wheel[node][0] for node in G_wheel.nodes()]
y_wheel = [pos_wheel[node][1] for node in G_wheel.nodes()]
axes[1].scatter(x_wheel, y_wheel, s=300, alpha=0)
axes[1].set_title("Wheel", fontsize=14)
axes[1].axis('off')
axes[1].set_aspect('equal')

# Plot complete
nx.draw_networkx_nodes(G_complete, pos_complete, ax=axes[2], node_color=values_complete, node_size=300, cmap=COLORMAP, vmin=0, vmax=1)
nx.draw_networkx_edges(G_complete, pos_complete, ax=axes[2], edge_color='gray', width=2, alpha=0.5)
# Add invisible scatterplot for all nodes
x_complete = [pos_complete[node][0] for node in G_complete.nodes()]
y_complete = [pos_complete[node][1] for node in G_complete.nodes()]
axes[2].scatter(x_complete, y_complete, s=300, alpha=0)
axes[2].set_title("Complete", fontsize=14)
axes[2].axis('off')
axes[2].set_aspect('equal')

#plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{NETWORK_KEY}_basic_network_types.pdf", dpi=300)
plt.show()

In [ ]:
pos_wheel

In [ ]:
import numpy as np
from matplotlib.ticker import ScalarFormatter

# Calculate degree distributions for original network only
networks = [
    ("Original", G_original),
]

print("=" * 60)
print("Degree Distribution Analysis")
print("=" * 60)

degree_data = {}

for name, G in networks:
    
    # Use out-degree and include all nodes (including 0 degree)
    degrees = [d for n, d in G.out_degree()]
    
    degree_data[name] = degrees
    
    # Gini coefficient
    gini = calculate_degree_gini(G, directed=True)
    
    print(f"\n{name}:")
    print(f"  Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    print(f"  Gini coefficient: {gini:.4f}")

# Plot symlog degree distributions
fig, axes = plt.subplots(1, 1, figsize=(6, 6))

# Get color from COLORMAP at 0.5
point_color = COLORMAP(0.5)

for idx, (name, G) in enumerate(networks):
    ax = axes
    degrees = degree_data[name]
    
    # Calculate degree distribution (including zeros)
    unique_degrees, counts = np.unique(degrees, return_counts=True)
    
    # Plot empirical distribution with larger points (using counts instead of probabilities)
    ax.plot(unique_degrees, counts, 'o', color=point_color, alpha=0.7, label='Empirical', markersize=10)
    
    ax.set_xscale('symlog')
    ax.set_yscale('symlog')
    ax.set_xlabel('Out-Degree (k)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
   # ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Format axes to show real numbers instead of scientific notation
    for axis in [ax.xaxis, ax.yaxis]:
        axis.set_major_formatter(ScalarFormatter())
        axis.set_minor_formatter(ScalarFormatter())
    
    ax.yaxis.set_minor_locator(plt.NullLocator())
    ax.yaxis.set_minor_formatter(plt.NullFormatter())
    ax.grid(True, alpha=1.)
    
    # Add Gini annotation towards the right side center of the plot
    ax.text(0.08, 0.08, f'Gini-coefficient = {calculate_degree_gini(G, directed=True):.3f}', 
            transform=ax.transAxes, fontsize=12, verticalalignment='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{NETWORK_KEY}_degree_distributions_loglog.pdf", dpi=300)
plt.show()

In [ ]:
try:
    import powerlaw
except ImportError:
    powerlaw = None
    print("Optional powerlaw package unavailable; plotting empirical degrees only.")
import numpy as np
from matplotlib.ticker import ScalarFormatter

# Calculate degree distributions for all three networks
networks = [
    ("Original", G_original),
    (G_variants[0][0], G_variants[0][1]),
    (G_variants[1][0], G_variants[1][1]),
]

print("=" * 60)
print("Degree Distribution Analysis with Power Law Statistics")
print("=" * 60)

degree_data = {}
powerlaw_results = {}

for name, G in networks:
    # Use out-degree and include all nodes (including 0 degree)
    degrees = [d for n, d in G.out_degree()]
    degree_data[name] = degrees
    
    # Gini coefficient
    gini = calculate_degree_gini(G, directed=True)
    
    # Power law analysis (filter out zeros for powerlaw package)
    degrees_nonzero = [d for d in degrees if d > 0]
    
    print(f"\n{name}:")
    print(f"  Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    print(f"  Gini coefficient: {gini:.4f}")
    
    if powerlaw is not None and len(degrees_nonzero) > 0:
        # Fit power law
        fit = powerlaw.Fit(degrees_nonzero, discrete=True, verbose=False)
        
        # Store results
        powerlaw_results[name] = {
            'fit': fit,
            'alpha': fit.power_law.alpha,
            'xmin': fit.power_law.xmin,
            'sigma': fit.power_law.sigma,
        }
        
        print(f"  Power Law Fit:")
        print(f"    Alpha (exponent): {fit.power_law.alpha:.4f} ± {fit.power_law.sigma:.4f}")
        print(f"    xmin: {fit.power_law.xmin}")
        
        # Compare power law to alternatives
        # Power law vs exponential
        R_exp, p_exp = fit.distribution_compare('power_law', 'exponential', normalized_ratio=True)
        print(f"  Comparison Tests (positive R favors power law):")
        print(f"    Power law vs Exponential: R = {R_exp:.4f}, p = {p_exp:.4f}")
        
        # Power law vs lognormal
        R_ln, p_ln = fit.distribution_compare('power_law', 'lognormal', normalized_ratio=True)
        print(f"    Power law vs Lognormal: R = {R_ln:.4f}, p = {p_ln:.4f}")
        
        # Power law vs truncated power law
        R_tpl, p_tpl = fit.distribution_compare('power_law', 'truncated_power_law', normalized_ratio=True)
        print(f"    Power law vs Truncated Power Law: R = {R_tpl:.4f}, p = {p_tpl:.4f}")
        
        # Lognormal vs exponential (to test if lognormal is preferred)
        R_ln_exp, p_ln_exp = fit.distribution_compare('lognormal', 'exponential', normalized_ratio=True)
        print(f"    Lognormal vs Exponential: R = {R_ln_exp:.4f}, p = {p_ln_exp:.4f}")
        
        # Lognormal vs truncated power law
        R_ln_tpl, p_ln_tpl = fit.distribution_compare('lognormal', 'truncated_power_law', normalized_ratio=True)
        print(f"    Lognormal vs Truncated Power Law: R = {R_ln_tpl:.4f}, p = {p_ln_tpl:.4f}")
        
        # Print lognormal fit parameters
        if fit.lognormal is not None:
            print(f"  Lognormal Fit:")
            print(f"    mu: {fit.lognormal.mu:.4f}")
            print(f"    sigma: {fit.lognormal.sigma:.4f}")
        
        powerlaw_results[name]['comparisons'] = {
            'exponential': (R_exp, p_exp),
            'lognormal': (R_ln, p_ln),
            'truncated_power_law': (R_tpl, p_tpl),
            'lognormal_vs_exponential': (R_ln_exp, p_ln_exp),
            'lognormal_vs_truncated_power_law': (R_ln_tpl, p_ln_tpl),
        }

# Plot degree distributions for all three networks
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Get color from COLORMAP at 0.5
point_color = COLORMAP(0.5)
powerlaw_color = 'red'

for idx, (name, G) in enumerate(networks):
    ax = axes[idx]
    degrees = degree_data[name]
    
    # Calculate degree distribution (including zeros) - using counts
    unique_degrees, counts = np.unique(degrees, return_counts=True)
    
    # Plot empirical distribution with larger points (using counts)
    ax.plot(unique_degrees, counts, 'o', color=point_color, alpha=0.7, label='Empirical', markersize=10)
    
    # Plot fitted power law line if available
    if name in powerlaw_results:
        fit = powerlaw_results[name]['fit']
        alpha = powerlaw_results[name]['alpha']
        xmin = powerlaw_results[name]['xmin']
        
        # Generate x values for the power law line (from xmin to max degree)
        x_powerlaw = np.linspace(xmin, max(degrees), 100)
        
        # Calculate the normalization constant C such that the line matches at xmin
        # Find the empirical count at or near xmin for scaling
        degrees_above_xmin = unique_degrees[unique_degrees >= xmin]
        counts_above_xmin = counts[unique_degrees >= xmin]
        
        if len(degrees_above_xmin) > 0:
            # Use the first point at or above xmin for normalization
            C = counts_above_xmin[0] * (degrees_above_xmin[0] ** alpha)
            y_powerlaw = C * (x_powerlaw ** (-alpha))
            
            ax.plot(x_powerlaw, y_powerlaw, '-', color=powerlaw_color, linewidth=2, 
                    label=f'Power Law (α={alpha:.2f})', alpha=0.8)
    
    ax.set_xscale('symlog')
    ax.set_yscale('symlog')
    ax.set_xlabel('Out-Degree (k)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'{name} Network', fontsize=14)
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Format axes to show real numbers instead of scientific notation
    for axis in [ax.xaxis, ax.yaxis]:
        axis.set_major_formatter(ScalarFormatter())
        axis.set_minor_formatter(ScalarFormatter())
    
    ax.yaxis.set_minor_locator(plt.NullLocator())
    ax.yaxis.set_minor_formatter(plt.NullFormatter())
    
    # Add Gini and power law annotations
    gini = calculate_degree_gini(G, directed=True)
    annotation_text = f'Gini = {gini:.3f}'
    if name in powerlaw_results:
        alpha = powerlaw_results[name]['alpha']
        annotation_text += f'\nα = {alpha:.2f}'
    
    ax.text(0.65, 0.5, annotation_text, 
            transform=ax.transAxes, fontsize=12, verticalalignment='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{NETWORK_KEY}_degree_distributions_all_networks_loglog.pdf", dpi=300)
plt.show()

In [ ]:
# Print nodes with highest out degree for each network
for name, G in [(NETWORK_TITLE, G_original)]:
    out_degrees = dict(G.out_degree())
    sorted_nodes = sorted(out_degrees.items(), key=lambda x: x[1], reverse=True)
    print(f"\n{name} Network - Top 10 nodes by out-degree:")
    for node, degree in sorted_nodes[:10]:
        print(f"  {G.nodes[node].get('author_id', node)}: {degree}")